In [ ]:
%config InlineBackend.figure_format = 'retina'

import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

from dotenv import load_dotenv
from matplotlib.patches import Patch
from openinference.semconv.trace import (
    OpenInferenceSpanKindValues,
    SpanAttributes
)
from sklearn.metrics import precision_score, recall_score, roc_auc_score

load_dotenv()

SPAN_KIND = SpanAttributes.OPENINFERENCE_SPAN_KIND
OUTPUT_VALUE = SpanAttributes.OUTPUT_VALUE
AGENT = OpenInferenceSpanKindValues.AGENT.value

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

In [ ]:
# FRAMES

original_input_path = "../data/frames/llm_frames_results_judged.csv"
original_input = pd.read_csv(original_input_path)
questions = original_input["Prompt"]
answers = original_input["Answer"]

base_path = "../logs/frames/llamacpp_qwen3_30b"
output_file = base_path.split("/")[-1]
max_folder = max(int(f) for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)))
data = []

for i in range(max_folder + 1):
    path = f"{base_path}/{i}"
    for run_idx in range(1):
        run_path = f"{path}/run_{run_idx}"
        try:
            summary = load_json(f"{run_path}/analysis/summary.json")
            step_summary = load_json(f"{run_path}/analysis/step_summary.json")
        except:
            print(f"Missing {i}")
            continue

        step_data = {}
        step_idx = 0
        for step in step_summary:
            if step["kind"] != "LLM":
                continue
            for k, v in step.items():
                step_data[f"step_{step_idx}_{k}"] = v
            step_idx += 1

        agent_output = summary["agent_output"]
        if agent_output is not None and "<think>" in agent_output and "</think>" in agent_output:
            summary["agent_output"] = agent_output[agent_output.find("</think>")+len("</think>"):].strip()

        data.append({
            "question": questions[i],
            "answer": answers[i],
            **summary,
            **step_data,
        })

data = pd.DataFrame.from_dict(data)
data.to_csv(f"../data/frames/profile_results_{output_file}.csv", index=False)

In [ ]:
# SimpleQA
original_input_path = "../data/simpleqa/llm_simpleqa_results_qwen3_1.7b_judged.csv"
original_input = pd.read_csv(original_input_path)
questions = original_input["problem"]
answers = original_input["answer"]

base_path = "../logs/simpleqa/llamacpp_qwen3_1.7b"
output_file = base_path.split("/")[-1]
max_folder = max(int(f) for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)))
data = []

for i in range(max_folder + 1):
    path = f"{base_path}/{i}"
    for run_idx in range(1):
        run_path = f"{path}/run_{run_idx}"
        summary = load_json(f"{run_path}/analysis/summary.json")
        step_summary = load_json(f"{run_path}/analysis/step_summary.json")

        step_data = {}
        step_idx = 0
        for step in step_summary:
            if step["kind"] != "LLM":
                continue
            for k, v in step.items():
                step_data[f"step_{step_idx}_{k}"] = v
            step_idx += 1

        agent_output = summary["agent_output"]
        if agent_output is not None and "<think>" in agent_output and "</think>" in agent_output:
            summary["agent_output"] = agent_output[agent_output.find("</think>")+len("</think>"):].strip()

        data.append({
            "question": questions[i],
            "answer": answers[i],
            **summary,
            **step_data,
        })

data = pd.DataFrame.from_dict(data)
data.to_csv(f"../data/simpleqa/profile_results_{output_file}.csv", index=False)